# Understanding classification models
## Confusion matrix, precision, recall and F1

This notebook uses **CIFAR-10**, the small colour images from notebook `03`, and asks one yes/no question
about every image: **is this a truck?** CIFAR-10 has ten classes with 1,000 test images each, so 1,000 of
the 10,000 test images are trucks and 9,000 are not. That is exactly the kind of imbalance in which a
single number, accuracy, can look good while the model does very little.

The example is meant for practice. The sections on thresholds and curves are optional.


## Setup

We use `numpy`, `pandas`, `matplotlib` and `scikit-learn` for the numbers and figures, and `torch` and
`torchvision` for the network and for loading CIFAR-10. The first run downloads CIFAR-10 (about 170 MB)
into `data/raw/`; if you did notebook `03`, it is already there.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    roc_auc_score,
    average_precision_score,
)
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

torch.manual_seed(42)                       # the same network and the same numbers on every run

# The data goes into data/raw of the project. In Colab the folder is simply created next to the notebook.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_dir = project_root / "data" / "raw"

train_set = datasets.CIFAR10(root=raw_dir, train=True, download=True)
test_set = datasets.CIFAR10(root=raw_dir, train=False, download=True)

print("classes:", train_set.classes)
print("training images:", len(train_set), "| test images:", len(test_set))


## 1. Why accuracy does not tell the whole story

Accuracy is the share of predictions that are correct. Suppose our model says 'not a truck' for every
image. It then gets 9,000 of the 10,000 test images right, but does not find a single truck.

We agree on the following: `1` means truck, `0` means anything else. The **positive class** here is
'truck'. Positive therefore means *the class we are looking for*.


In [ ]:
# the class number of "truck"
truck = train_set.class_to_idx["truck"]                          
print(f"truck: {truck}")
# One label per image: 1 if it is a truck, 0 if it is anything else.
y_train = (np.array(train_set.targets) == truck).astype(int)
y_true = (np.array(test_set.targets) == truck).astype(int)       # y_true: the truth for the test images

# This "model" says "not a truck" for every image.
y_baseline = np.zeros(len(y_true), dtype=int)

trucks_found = ((y_true == 1) & (y_baseline == 1)).sum()

print(f"Number of test images: {len(y_true)}")
print(f"Number that are trucks: {y_true.sum()}")
print(f"Accuracy: {accuracy_score(y_true, y_baseline):.1%}")
print(f"Trucks found: {trucks_found} of {y_true.sum()}")


In [ ]:
# Ten random test images, with their real class above them: trucks are about one in ten.
rng = np.random.default_rng(0)
sample = rng.choice(len(test_set), size=10, replace=False)

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for ax, index in zip(axes, sample):
    ax.imshow(test_set.data[index])                              # data holds the pixels: (10000, 32, 32, 3)
    ax.set_title(test_set.classes[test_set.targets[index]], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


**Think about it:** would you decide to use this model based on that 90%? What information is missing?


## 2. The confusion matrix

To say anything about a real model we need one. We train a small convolutional network in PyTorch, as in
notebook `03`, to answer 'truck or not': two `Conv2d` layers with pooling, then one `Linear` layer with two
outputs, one score for 'not a truck' and one for 'truck'. Training takes about a minute on a CPU. The
network is not the point of this notebook; its predictions are.


In [ ]:
# Images as PyTorch wants them: (batch, channels, height, width), with values between 0 and 1.
def to_tensor(images):
    return torch.tensor(images).permute(0, 3, 1, 2).float() / 255      # (N, 32, 32, 3) -> (N, 3, 32, 32)


X_train = to_tensor(train_set.data)
X_test = to_tensor(test_set.data)
train_loader = DataLoader(TensorDataset(X_train, torch.tensor(y_train)), batch_size=128, shuffle=True)

network = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 3 x 32 x 32  -> 16 x 16 x 16
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    #              -> 32 x 8 x 8
    nn.Flatten(),                                                               #              -> 2048 numbers
    nn.Linear(32 * 8 * 8, 2),                                                   # 2 scores: not a truck, truck
)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)

for epoch in range(3):
    network.train()
    total_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        loss = loss_function(network(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f"epoch {epoch + 1}: training loss {total_loss / len(train_loader.dataset):.4f}")

network.eval()
with torch.no_grad():                                    # no gradients needed to predict
    probabilities = torch.softmax(network(X_test), dim=1)       # 2 numbers per image that add up to 1

scores = probabilities[:, 1].numpy()                     # the score for "truck", used in the optional sections
y_pred = probabilities.argmax(dim=1).numpy()             # the label: 1 = "truck", 0 = "not a truck"

print(f"accuracy of the model: {accuracy_score(y_true, y_pred):.1%}  (always 'not a truck': {accuracy_score(y_true, y_baseline):.1%})")


That is a little better than the baseline. Is it also useful? The confusion matrix shows which predictions
are right and which are wrong. Every image ends up in exactly one cell.

| Abbreviation | Meaning in this example |
|---|---|
| **TP**, true positive | A truck, correctly predicted as a truck. |
| **FN**, false negative | A truck, but predicted as not a truck. A missed truck. |
| **FP**, false positive | Not a truck, but predicted as a truck. A false alarm. |
| **TN**, true negative | Not a truck, correctly predicted as not a truck. |


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

matrix_table = pd.DataFrame(
    cm,
    index=["Actually not a truck (0)", "Actually a truck (1)"],
    columns=["Predicted not a truck (0)", "Predicted truck (1)"],
)

display(matrix_table)

tn, fp, fn, tp = cm.ravel()
print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")


**Mind the order:** with `labels=[0, 1]`, scikit-learn uses the layout `[[TN, FP], [FN, TP]]`. The rows give
the actual class, the columns the predicted class. Slides and textbooks often put the positive class first
instead, so check which layout you are looking at.

The next cell turns it into a figure.


In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.6))

ConfusionMatrixDisplay(
    cm,
    display_labels=["Not a truck", "Truck"]
).plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

ax.set_xlabel("Model prediction")
ax.set_ylabel("Actual situation")
ax.set_title("Is it a truck? 10,000 test images")

plt.tight_layout()
plt.show()


### Look at the mistakes

A number tells you how many mistakes there are. The images tell you what kind. The first cell counts the
false alarms per real class; the second shows a few false alarms and a few missed trucks.


In [ ]:
real_class = np.array(test_set.classes)[np.array(test_set.targets)]      # the class name of every test image

false_alarms = (y_true == 0) & (y_pred == 1)
print("false alarms, by what the image really is:")
print(pd.Series(real_class[false_alarms]).value_counts().to_string())


In [ ]:
def show_images(indices, title):
    fig, axes = plt.subplots(1, len(indices), figsize=(1.7 * len(indices), 2.2))
    for ax, index in zip(axes, indices):
        ax.imshow(test_set.data[index])
        ax.set_title(real_class[index], fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_images(np.where(false_alarms)[0][:8], "False alarms: predicted truck, but it is not")
show_images(np.where((y_true == 1) & (y_pred == 0))[0][:8], "Missed trucks: predicted not a truck, but it is one")


Look at which classes show up most among the false alarms: automobiles, which share wheels, a road and a
boxy shape with a truck. The metrics in the next sections summarise the four cells; the images remind you
what the cells stand for.


## 3. Accuracy, precision, recall and F1

### Accuracy

What share of all predictions is correct?

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

### Precision

When the model says 'truck', how often is that right?

$$\text{Precision} = \frac{TP}{TP + FP}$$

### Recall

What share of all actual trucks does the model find?

$$\text{Recall} = \frac{TP}{TP + FN}$$

### F1

F1 combines precision and recall using the harmonic mean. A very low value for either one pulls F1 down.

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2\,TP}{2\,TP + FP + FN}$$


In [ ]:
results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Score": [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),         # precision, recall and F1 look at class 1: "truck"
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred),
    ],
})

display(results.round(3))

print()
print(f"The model called {tp + fp} images a truck, and {tp} of them really were.")
print(f"There are {tp + fn} trucks in the test set, and the model found {tp} of them.")


The model has a high accuracy, and when it says 'truck' it is usually right, but it finds only about three
in ten of the trucks. Accuracy hides all of that. Which of the numbers is "the" quality of the model
depends on the question you ask.

Precision, recall and F1 look at one class, the positive class. In scikit-learn that is `pos_label`, and
its default is `1`: 'truck'.


### What if the model never raises the alarm?

Then precision is undefined, because $TP + FP$ is zero. Our baseline from section 1, 'not a truck' for
everything, is exactly that model. Recall is 0 for it, and so is F1.


In [ ]:
print("baseline: alarms raised =", y_baseline.sum())
print("baseline: recall        =", recall_score(y_true, y_baseline))
print("baseline: precision     =", precision_score(y_true, y_baseline, zero_division=0), " (0 divided by 0, filled in as 0)")
print("baseline: F1            =", f1_score(y_true, y_baseline, zero_division=0))


Some software fills in a 0 for undefined precision. That is a software choice. In addition, F1 does not take
TN into account and weighs precision and recall equally. So also look at the individual metrics and at the
consequences of errors.


## 4. Which metric fits the goal?

It depends on what a mistake costs. Say 'truck' is the thing you are looking for:

- A system that counts heavy vehicles for a road toll: a missed truck (an FN) is lost income, so you pay
  extra attention to **recall**.
- A system that fines a driver automatically: a false alarm (an FP) means a wrong fine for someone who did
  nothing wrong, so you pay extra attention to **precision**.

In screening, missing a sick person can weigh heavily. With a spam filter, hiding an important email can
weigh heavily. The right trade-off depends on the context.


## Try it yourself: flip the question

Take the same model and the same predictions, but now call **'not a truck' the positive class**. Nothing
is retrained: only the label you are looking for changes.

1. What are TP, FP, FN and TN now? The four counts from section 2 are still there (`tn`, `fp`, `fn`, `tp`),
   but every one of them is now a different cell. Which is which?
2. Get precision, recall and F1 for the new positive class from scikit-learn. Look at the `pos_label`
   argument.
3. The model looked mediocre before. Why does it look so much better now?

Replace the zeros below with your answers. The `assert` lines tell you whether they are right.


In [ ]:
# The positive class is now 0 ("not a truck").

# 1. The four counts. Hint: which of the old cells (tn, fp, fn, tp) is "not a truck, predicted as not a truck"?
my_tp = 0
my_fp = 0
my_fn = 0
my_tn = 0

assert my_tp == ((y_true == 0) & (y_pred == 0)).sum(), "TP: not a truck AND predicted as not a truck"
assert my_fp == ((y_true == 1) & (y_pred == 0)).sum(), "FP: predicted 'not a truck', but it IS a truck"
assert my_fn == ((y_true == 0) & (y_pred == 1)).sum(), "FN: not a truck, but predicted as a truck"
assert my_tn == ((y_true == 1) & (y_pred == 1)).sum(), "TN: trucks that were predicted as a truck"
print("The four counts are right.")

# 2. The three metrics for the new positive class, from scikit-learn.
my_precision = 0.0
my_recall = 0.0
my_f1 = 0.0

assert my_precision > 0.9 and my_recall > 0.9 and my_f1 > 0.9, (
    "this question should look very good for this model: did you set pos_label=0 in all three functions?")
print("Precision, recall and F1 are right.")


## 5. Optional: the threshold

Many models first produce a score. Here we predict 'truck' if `score >= threshold`. The `predict` method
used the default threshold of 0.5. A score between 0 and 1 is not necessarily a calibrated probability.

We compare five thresholds on the scores of the test images.


In [ ]:
def metrics_at(threshold):
    predicted = (scores >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "Threshold": threshold,
        "Flagged": int(predicted.sum()),
        "TP": tp_t,
        "FP": fp_t,
        "FN": fn_t,
        "TN": tn_t,
        "Precision": precision_score(y_true, predicted, zero_division=0),
        "Recall": recall_score(y_true, predicted),
        "F1": f1_score(y_true, predicted, zero_division=0),
    }


threshold_table = pd.DataFrame([metrics_at(t) for t in [0.5, 0.3, 0.2, 0.1, 0.05]])

view = threshold_table.copy()
for column in ["Precision", "Recall"]:
    view[column] = view[column].map(lambda value: f"{value:.0%}")
view["F1"] = view["F1"].round(2)
display(view)


A lower threshold raises the alarm sooner. Recall does not decrease when you lower the threshold. Precision
can decrease, but does not have to drop neatly at every step. F1 is highest somewhere in the middle, and
not at the default of 0.5.

**Practice question:** you want at least 75% recall of the trucks. Which of the five tested thresholds do
you choose if you also want the highest precision?

The cell below draws all thresholds from 0.05 to 0.95.


In [ ]:
curve = pd.DataFrame([metrics_at(t) for t in np.arange(0.05, 1.0, 0.05)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["Threshold"], curve["Precision"], marker="o", label="precision")
ax.plot(curve["Threshold"], curve["Recall"], marker="o", label="recall")
ax.plot(curve["Threshold"], curve["F1"], marker="o", label="F1")
ax.axvline(0.5, color="black", linestyle=":", label="the default 0.5")
ax.set(xlabel="threshold", ylabel="score", title="One model, every threshold")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Optional: PR and ROC curves

A threshold gives a single operating point. A curve shows what happens at different thresholds. The code
below uses all the scores of the test images, not just the five thresholds from the table.

- The **precision-recall curve** puts recall on the x-axis and precision on the y-axis.
- The **ROC curve** puts the false positive rate on the x-axis and recall on the y-axis.

The false positive rate is:

$$FPR = \frac{FP}{FP + TN}$$

This is the share of images that are not a truck and wrongly get an alarm.

When positive cases are rare, a small false positive rate can still go together with many false alarms. The
PR curve helps to judge how trustworthy the alarms are.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

# Precision-recall
PrecisionRecallDisplay.from_predictions(y_true, scores, ax=axes[0])

axes[0].axhline(
    y_true.mean(),
    color="gray",
    linestyle=":",
    label=f"Positive class: {y_true.mean():.0%}"
)

axes[0].set(
    title="Precision-recall",
    xlabel="Recall",
    ylabel="Precision"
)
axes[0].legend(loc="upper right")

# ROC
RocCurveDisplay.from_predictions(y_true, scores, ax=axes[1])

axes[1].plot(
    [0, 1], [0, 1],
    "--",
    color="gray",
    label="Random ranking"
)

axes[1].set(
    title="ROC",
    xlabel="False positive rate",
    ylabel="Recall (TPR)"
)
axes[1].legend(loc="lower right")

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc_auc_score(y_true, scores):.3f}   (random ranking: 0.500)")
print(f"Average precision (AP): {average_precision_score(y_true, scores):.3f}   (random ranking: {y_true.mean():.3f})")


**ROC-AUC** summarises the ranking: it is the chance that a random truck gets a higher score than a random
non-truck. Here that is about 0.93, which sounds excellent. The **average precision** is about 0.67, against
a random baseline of 0.10 (not 0.5). Compare each with its own baseline, and the PR curve looks a lot less
comfortable than the ROC curve.

The reason is the denominator. The false positive rate divides by the 9,000 images that are not a truck, so
hundreds of false alarms barely move it. Take the threshold 0.1 from the table: its point on the ROC curve
is at a false positive rate of about 0.09 and a recall of about 0.77, which looks excellent. But fewer than
half of its alarms are real, because precision divides by the alarms you raised, and there are many. **When
the positive class is rare, ROC-AUC is mostly measured on the class you do not care about.**

A high AUC does not guarantee a useful alarm at the chosen threshold.

**Average precision (AP)** summarises the PR curve with a weighted sum over recall steps. It is not the same
as the trapezoidal area under a drawn PR line. You can skip this detail on a first read.


## 7. Short self-test

1. Explain precision and recall without a formula.
2. Why can 90% accuracy still be insufficient for finding trucks?
3. What happens to recall if you predict every image as a truck? And to precision?
4. Why is precision undefined when there are no alarms?
5. Name a situation in which a missed case weighs more heavily than a false alarm.


## Answers

### Flip the question

The old cells move like this (in our run; yours may differ by a few):

- TP = 8,933: this was the old TN, a non-truck correctly predicted as not a truck
- FP = 686: this was the old FN, a truck that was predicted as not a truck
- FN = 67: this was the old FP, a non-truck that was predicted as a truck
- TN = 314: this was the old TP, a truck correctly predicted as a truck

```python
my_tp = tn
my_fp = fn
my_fn = fp
my_tn = tp

my_precision = precision_score(y_true, y_pred, pos_label=0)
my_recall = recall_score(y_true, y_pred, pos_label=0)
my_f1 = f1_score(y_true, y_pred, pos_label=0)
```

That makes precision about 93%, recall about 99% and F1 about 0.96. The model did not change: the *question*
did. Now the positive class is the common one, which is easy to get right, so every metric looks good. The
metrics always belong to a positive class, and they only mean something if that is the class you are
looking for.

### Threshold question

Look in your table for the rows with a recall of at least 75%, and take the one with the highest
precision. In our run that is **0.1**: recall about 77%, precision about 48%. At 0.2 the recall is only
about 62%, so that is too low. At 0.05 the recall is higher, about 87%, but the precision falls further, to
about 37%. Choosing the threshold from a requirement like this is how a threshold should be chosen; the
default 0.5 gives a recall of only about 31%.

### Self-test

1. Precision: how often is an alarm justified? Recall: what share of the actual positive cases do you find?
2. A model that says 'not a truck' for everything has 90% accuracy and finds no truck at all. With 1,000
   trucks among 10,000 images, accuracy mostly measures the 9,000 images you do not care about.
3. Recall becomes 100%, because nothing is missed. Precision falls to the share of trucks, 10%, because
   nine out of ten alarms are false.
4. Without alarms, $TP + FP = 0$. Precision then has a denominator of zero.
5. One possible answer is screening where you want to miss as few sick people as possible, or a truck
   counter that earns money per truck. The choice depends on the consequences of errors.

**Key takeaway:** look at the confusion matrix and choose metrics that fit the goal and the consequences of
errors.
